In [ ]:
# Notice we are now using google-genai instead of google-generativeai
!pip uninstall -y playwright-stealth tf-playwright-stealth
!pip install playwright-stealth==2.0.2 -q

!playwright install chromium
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 19.0 MB/s eta 0:00:00
(node:11092) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 64.6s167.3 MiB [] 0% 38.8s167.3 MiB [] 0% 28.0s167.3 MiB [] 0% 30.7s167.3 MiB [] 0% 25.8s167.3 MiB [] 0% 18.8s167.3 MiB [] 0% 14.2s167.3 MiB [] 1% 9.8s167.3 MiB [] 2% 6.9s167.3 MiB [] 3% 5.8s167.3 MiB [] 3% 5.0s167.3 MiB [] 3% 5.2s167.3 MiB [] 4% 5.5s167.3 MiB [] 4% 5.2s167.3 MiB [] 5% 4.4s167.3 MiB [] 6% 3.9s167.3 MiB [] 7% 3.7s167.3 MiB [] 8% 3.5s167.3 MiB [] 9% 3.4s167.3 MiB [] 9% 3.3s167.3 MiB [] 10% 3.2s167.3 MiB [] 11% 3.1s167.3 MiB [] 11% 3.0s167.3 MiB [] 12% 2.9s167.3 MiB [] 13% 2.9s167.3 MiB [] 14% 2.8s167.3 MiB [] 14% 2.7s167.3 MiB [] 15% 2.6s167.3 MiB [] 16% 2.6s167.

In [ ]:
from google import genai
from google.colab import userdata
import json

# Fetch the API key you saved in the Secrets tab
api_key = userdata.get('GEMINI_API_KEY')

# Initialize the NEW client
client = genai.Client(api_key=api_key)

def get_outfit_keywords(vibe_description):
    """Translates a vibe into a JSON list of Depop search keywords."""

    print(f"🧠 Asking AI to analyse vibe: '{vibe_description}'...")

    prompt = f"""
    You are a fashion stylist expert in finding specific aesthetics on second-hand clothing apps like Depop.
    A user wants an outfit for this theme/vibe: "{vibe_description}"

    Translate this vibe into exactly 2 concrete clothing search queries, and for each query an associated clothing category.
    Use terms people actually use on Depop (e.g., "Y2K oversized graphic tee", "baggy light wash jeans", "distressed leather jacket", "Oversized Blazer").
    Try to avoid subjective terms such as 'Quirky' or 'Edgy' unless they directly correlate to a specific trend or aesthetic.
    Do NOT include the main categories 'Womenswear', 'Menswear', 'Jewellery', 'Beauty', or 'Kids' as a prefix in your output categories. For example, if an item is in 'Womenswear/Tops/T-shirts', output only 'Tops/T-shirts'. Your output categories should always reflect the sub-category path starting from the second level.

    Respond ONLY with a raw JSON list of strings. Do not include markdown formatting or backticks.
    Example output: ["baggy yellow hoodie: Tops/Jumpers", "oversized graphic tee y2k: Tops/T-shirts", "baggy light wash jeans: Bottoms/Jeans", "Vintage wool jumper: Tops/Jumpers"]
    """
    depop_taxonomy = """

    MAIN CATEGORIES & SUB-CATEGORIES:

    1. Womenswear
      - Tops: T-shirts, Crop Tops, Vests, Tank Tops, Blouses, Shirts, Sweatshirts, Hoodies, Bodysuits
      - Bottoms: Jeans, Trousers, Skirts, Shorts, Leggings, Joggers
      - Dresses: Mini, Midi, Maxi, Summer, Prom, Evening
      - Outerwear: Coats, Jackets, Blazers, Denim Jackets, Leather Jackets, Puffers
      - Shoes: Sneakers, Boots, Heels, Loafers, Sandals, Flats, Platforms
      - Accessories: Bags, Hats, Sunglasses, Belts, Scarves, Hair Accessories
      - Lingerie & Nightwear: Lingerie, Loungewear, Sleepwear

    2. Menswear
      - Tops: T-shirts, Polo Shirts, Shirts, Sweatshirts, Hoodies, Vests
      - Bottoms: Jeans, Trousers, Shorts, Joggers
      - Outerwear: Coats, Jackets, Blazers, Bombers, Windbreakers
      - Shoes: Sneakers, Boots, Loafers, Formal Shoes, Sandals
      - Accessories: Bags, Hats, Sunglasses, Belts, Watches, Jewelry

    3. Jewellery (Often its own top-level in Meganav)
      - Necklaces, Rings, Earrings, Bracelets, Brooches

    4. Beauty (Newer top-level category)
      - Makeup, Skincare, Fragrance, Haircare

    5. Kids
      - Tops, Bottoms, Dresses, Outerwear, Shoes
    """
    # Append the taxonomy to the prompt
    prompt += "\n\nHere is the Depop taxonomy for your reference:\n" + depop_taxonomy

    try:
        # Generate content using the new SDK syntax and a current model
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
        )

        # Clean up and parse the JSON response
        clean_text = response.text.strip().replace('```json', '').replace('```', '')
        keywords = json.loads(clean_text)
        return keywords

    except Exception as e:
        print(f"⚠️ Error parsing AI response: {e}")
        return []

In [ ]:
# Test it!
keywords = get_outfit_keywords("the Year 2016")
print("Success! Generated keywords:", keywords)

🧠 Asking AI to analyse vibe: 'the Year 2016'...
Success! Generated keywords: ['bomber jacket: Outerwear/Jackets', 'distressed skinny jeans: Bottoms/Jeans']


In [ ]:
import nest_asyncio
import asyncio
import random
import pandas as pd
from playwright.async_api import async_playwright
from playwright_stealth import Stealth
import urllib.parse

# Apply fix for nested event loops in Colab
nest_asyncio.apply()

async def search_depop_vibe(keyword, location="GB", category=None, size_slug=None, max_items=3, gender=None):
    results = []
    stealth = Stealth()

    # Map location codes to locale and timezone for robust spoofing
    location_settings = {
        "GB": {"locale": "en-GB", "timezone_id": "Europe/London", "accept_language": "en-GB,en;q=0.9"},
        "US": {"locale": "en-US", "timezone_id": "America/New_York", "accept_language": "en-US,en;q=0.9"},
        # Add more locations as needed
    }
    current_location_settings = location_settings.get(location.upper(), location_settings["GB"]) # Default to GB

    async with async_playwright() as p:
        # 1. Launch Browser
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-gpu", "--disable-dev-shm-usage"]
        )

        # 2. Set Context for the specified location dynamically
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            locale=current_location_settings["locale"],
            timezone_id=current_location_settings["timezone_id"],
            extra_http_headers={"Accept-Language": current_location_settings["accept_language"]}
        )

        # 3. INJECT COOKIES for the specified location dynamically
        await context.add_cookies([
            {'name': 'geoland', 'value': location.upper(), 'domain': '.depop.com', 'path': '/'},
            {'name': 'country_code', 'value': location.upper(), 'domain': '.depop.com', 'path': '/'},
            {'name': 'language_code', 'value': current_location_settings["locale"].split('-')[0], 'domain': '.depop.com', 'path': '/'}
        ])

        # Protect the context with Stealth
        await stealth.apply_stealth_async(context)
        page = await context.new_page()

        categories_to_try = []
        if category:
            if gender == 'Womenswear':
                categories_to_add = [f"Womenswear/{category}"]
            elif gender == 'Menswear':
                categories_to_add = [f"Menswear/{category}"]
            else:
                # If gender is not specified, try both, then the generic category
                categories_to_add = [f"Womenswear/{category}", f"Menswear/{category}", category]
            # Filter out None or empty strings from the list of categories to add
            categories_to_try.extend([c for c in categories_to_add if c])
        else:
            categories_to_try.append(None)

        print(f"🕵️‍♂️ Searching: '{keyword}' on Depop {location.upper()}...")

        final_results = []
        # Ensure we don't try duplicate categories if, for example, category was already 'Womenswear/Tops'
        categories_to_try = list(dict.fromkeys(categories_to_try)) # Remove duplicates while preserving order

        for cat_attempt in categories_to_try:
            if len(final_results) >= max_items:
                break

            # Dynamically construct the base URL using the location parameter
            current_search_url = f"https://www.depop.com/{location.lower()}/search/?q={urllib.parse.quote(keyword)}"

            if cat_attempt:
                current_search_url += f"&categories={cat_attempt}"
            if size_slug:
                current_search_url += f"&sizes={size_slug}"

            print(f"   Attempting URL: {current_search_url}")

            try:
                retries = 3
                for i in range(retries):
                    try:
                        await page.goto(current_search_url, wait_until="domcontentloaded", timeout=60000)
                        await asyncio.sleep(random.uniform(2, 4))
                        break
                    except Exception as e:
                        print(f"   ⚠️ Attempt {i+1} failed: {e}")
                        if i == retries - 1: raise
                        await asyncio.sleep(5)

                # --- SCRAPE RESULTS ---
                await page.wait_for_selector("a[href*='/products/']", timeout=15000)
                items = await page.query_selector_all("a[href*='/products/']")

                items_to_add = max_items - len(final_results)
                for item in items[:items_to_add]:
                    link = await item.get_attribute('href')
                    img_element = await item.query_selector("img")

                    if img_element:
                        img_url = await img_element.get_attribute('src')

                        # Attempt to get the product title more robustly
                        item_title = "Vintage Find"
                        # Try different common selectors for product title within the <a> tag
                        title_selectors = [
                            "h2", # Common for product titles
                            "p[data-testid*='product-title']", # Depop specific, often data-testid
                            "p[class*='ProductCard__Title']", # Depop specific, class names
                            "div[class*='Text__StyledText']", # Depop specific, another class
                            "p", # General paragraph tag
                            "div" # General div tag
                        ]
                        for selector in title_selectors:
                            title_elem = await item.query_selector(selector)
                            if title_elem:
                                raw_title_text = await title_elem.inner_text()
                                if raw_title_text.strip():
                                    item_title = raw_title_text.strip()
                                    break

                        # Fallback to alt text if available and title still generic
                        if item_title == "Vintage Find" and img_element:
                            alt_text = await img_element.get_attribute('alt')
                            if alt_text and alt_text.strip():
                                item_title = alt_text.strip()

                        # Final fallback to item.inner_text() if all else fails, or if it contains useful info
                        if item_title == "Vintage Find":
                            full_item_text = await item.inner_text()
                            if full_item_text.strip():
                                # Take the first non-empty line as a fallback item title
                                for line in full_item_text.split('\n'):
                                    if line.strip():
                                        item_title = line.strip()
                                        break

                        # Join correctly with Depop base for the specified location
                        absolute_link = urllib.parse.urljoin(f"https://www.depop.com/{location.lower()}/", link)

                        final_results.append({
                            "Keyword": keyword,
                            "Item": item_title,
                            "Link": absolute_link,
                            "Image": img_url,
                            "Raw_Text": await item.inner_text(), # Keep raw text for debugging
                            "Category_Attempted": cat_attempt # Add the category attempted for debugging
                        })

                if final_results:
                    print(f"   ✅ Found {len(final_results)} items.")

            except Exception as e:
                print(f"   ⚠️ Scraping failed: {e}")

        await browser.close()

    return final_results

In [ ]:
async def main(vibe, location='GB', size_slug=None, gender=None):
    print("====================================")
    print(f"👗 TARGET VIBE: {vibe}")
    print(f"📍 TARGET LOCATION: {location}")
    if size_slug: print(f"📏 TARGET SIZE: {size_slug}")
    if gender: print(f"🚻 TARGET GENDER: {gender}")
    print("====================================\n")

    # 1. Get Keywords from AI
    keywords_with_categories = get_outfit_keywords(vibe)
    print(f"\n✅ AI Suggested Search Terms: {keywords_with_categories}\n")

    all_outfit_pieces = []

    # 2. Search Depop for each keyword
    for item_str in keywords_with_categories:
        # Parse the keyword and category from the combined string
        if ':' in item_str:
            keyword, category = item_str.split(':', 1)
            keyword = keyword.strip()
            category = category.strip()
        else:
            keyword = item_str.strip()
            category = None # No category provided

        # Crucial delay to avoid bot detection
        await asyncio.sleep(random.uniform(2.0, 5.0))

        # Call the correct function name 'search_depop_vibe' and pass location, category, size_slug, and gender
        items = await search_depop_vibe(keyword, location=location, category=category, size_slug=size_slug, max_items=1, gender=gender)

        if items:
            all_outfit_pieces.extend(items)
            print(f"✅ Found {len(items)} items for '{keyword}'.")
        else:
            print(f"❌ No items found for '{keyword}'.")

    print("\n====================================")
    print("🎉 SEARCH COMPLETE! 🎉")
    print("====================================")

    return all_outfit_pieces

In [ ]:
# --- EXECUTE THE PROJECT ---
vibe_request = "The year 2016"
location_request = "GB" # Set your desired location here (e.g., 'GB', 'US')
size_request = "10-20.4-UK" # Example size slug for UK size 10 (modify as needed)
gender_request = "Womenswear" # Can be 'Womenswear', 'Menswear', or None

# In Colab, we can often just await the main function directly
# because nest_asyncio is applied.
try:
    found_items = await main(vibe_request, location=location_request, size_slug=size_request, gender=gender_request)

    # Create the DataFrame
    df = pd.DataFrame(found_items)

    if not df.empty:
        # Make the links clickable in the table
        from IPython.display import HTML
        display(HTML(df.to_html(render_links=True, escape=False)))
    else:
        print("No items were found. Try a different vibe or check your connection.")

except Exception as e:
    print(f"An error occurred during execution: {e}")

👗 TARGET VIBE: The year 2016
📍 TARGET LOCATION: GB
📏 TARGET SIZE: 10-20.4-UK
🚻 TARGET GENDER: Womenswear

🧠 Asking AI to analyse vibe: 'The year 2016'...

✅ AI Suggested Search Terms: ['satin bomber jacket: Outerwear/Jackets', 'high waisted ripped skinny jeans: Bottoms/Jeans']

🕵️‍♂️ Searching: 'satin bomber jacket' on Depop GB...
   Attempting URL: https://www.depop.com/gb/search/?q=satin%20bomber%20jacket&categories=Womenswear/Outerwear/Jackets&sizes=10-20.4-UK
   ✅ Found 1 items.
✅ Found 1 items for 'satin bomber jacket'.


In [ ]:
# Setup your parameters
vibe_query = "y2k jeans"
target_category = "ladies-bottoms" # The category required to unlock the size filter
target_size = "10-20.4-UK"        # The specific size 10 slug

# Run the search
results = await search_depop_vibe(
    keyword=vibe_query,
    category=target_category,
    size_slug=target_size,
    max_items=3 # Define max_items here
)
print(results)

🕵️‍♂️ Searching: 'y2k jeans' on Depop GB...
   Attempting URL: https://www.depop.com/gb/search/?q=y2k%20jeans&categories=Womenswear/ladies-bottoms&sizes=10-20.4-UK
   ✅ Found 3 items.
[{'Keyword': 'y2k jeans', 'Item': 'Vintage Find', 'Link': 'https://www.depop.com/products/imsofrixkenballer-vintage-american-eagle-distressed-blue-d59e/', 'Image': 'https://media-photos.depop.com/b1/44715695/3603671473_90d311ade6364828a4fb2113827bfbdd/P10.jpg', 'Raw_Text': '', 'Category_Attempted': 'Womenswear/ladies-bottoms'}, {'Keyword': 'y2k jeans', 'Item': 'Vintage Find', 'Link': 'https://www.depop.com/products/morganb6rt-y2k-vigoss-low-rise-bootcut-denim-791d/', 'Image': 'https://media-photos.depop.com/r1/25114730/3596726167_fc50137f8aa143b2a0d3ff1d9dcf36ea/P10.jpg', 'Raw_Text': '', 'Category_Attempted': 'Womenswear/ladies-bottoms'}, {'Keyword': 'y2k jeans', 'Item': 'Vintage Find', 'Link': 'https://www.depop.com/products/andresses-thats-so-fetch-light-blue-bd4b/', 'Image': 'https://media-photos.depop